# IPW Effect Estimation and Post-Weighting Balance

## Scenario
Following the propensity modeling step, apply inverse probability weighting (IPW)
to estimate the causal effect of the "first box free" trial offer on conversion
and revenue. Verify that IPW reduces covariate imbalance and compare adjusted
estimates to naive estimates to quantify selection bias.

## Data: `mealkit_trial_adoption.csv`


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('Exercise_Data')
if not DATA_DIR.exists():
    DATA_DIR = Path('/content/drive/Othercomputers/My PC/Desktop/All_Work_Related/Udacity/Applied_Statistics_ND/Exercises/Udacity Exercises Redone/')

ADOPT_PATH = DATA_DIR / 'mealkit_trial_adoption.csv'
from sklearn.linear_model import LogisticRegression

df = pd.read_csv(ADOPT_PATH)
# Refit propensity model (same setup as propensity exercise)
df_enc = df.fillna({'region': 'Unknown'})
df_enc = pd.get_dummies(df_enc, columns=['region','device'], drop_first=True)
encoded_cat_cols = [c for c in df_enc.columns if c.startswith(('region_','device_'))]
numeric_covs = ['age','household_size','prior_orders','weekly_app_sessions']
all_covs = numeric_covs + encoded_cat_cols
X = df_enc[all_covs].astype(float)
y = df_enc['trial_offer_sent']
prop_model = LogisticRegression(max_iter=500, random_state=42)
prop_model.fit(X, y)
df_enc['propensity'] = prop_model.predict_proba(X)[:, 1]
print(f"Propensity range: {df_enc['propensity'].min():.4f} – {df_enc['propensity'].max():.4f}")

Propensity range: 0.0595 – 0.9179


## Step 1: Compute IPW Weights

In [3]:
df_enc['ipw'] = np.where(
    df_enc['trial_offer_sent'] == 1,
    1.0 / df_enc['propensity'],
    1.0 / (1.0 - df_enc['propensity'])
)
print("IPW weight summary:")
print(df_enc['ipw'].describe().round(3))

IPW weight summary:
count    8000.000
mean        2.000
std         1.059
min         1.068
25%         1.358
50%         1.652
75%         2.230
max        16.796
Name: ipw, dtype: float64


## Step 2: Post-IPW Balance Check (SMD)

In [4]:
treated = df_enc[df_enc['trial_offer_sent'] == 1]
control = df_enc[df_enc['trial_offer_sent'] == 0]

def smd_raw(col, t, c):
    pooled_std = np.sqrt((t[col].var() + c[col].var()) / 2 + 1e-9)
    return (t[col].mean() - c[col].mean()) / pooled_std

def smd_ipw(col, df_full):
    t = df_full[df_full['trial_offer_sent'] == 1]
    c = df_full[df_full['trial_offer_sent'] == 0]
    mt = np.average(t[col], weights=t['ipw'])
    mc = np.average(c[col], weights=c['ipw'])
    pooled_std = np.sqrt((t[col].var() + c[col].var()) / 2 + 1e-9)
    return (mt - mc) / pooled_std

smd_before = pd.Series({v: smd_raw(v, treated, control) for v in numeric_covs})
smd_after  = pd.Series({v: smd_ipw(v, df_enc) for v in numeric_covs})
balance = pd.DataFrame({'SMD Before': smd_before, 'SMD After (IPW)': smd_after})
print("Covariate Balance (SMD):")
print(balance.round(4))
print(f"\nMax |SMD| before: {balance['SMD Before'].abs().max():.4f}")
print(f"Max |SMD| after:  {balance['SMD After (IPW)'].abs().max():.4f}")

Covariate Balance (SMD):
                     SMD Before  SMD After (IPW)
age                     -0.2082          -0.0034
household_size           0.6063          -0.0009
prior_orders             0.5242          -0.0021
weekly_app_sessions      0.1247           0.0013

Max |SMD| before: 0.6063
Max |SMD| after:  0.0034


## Step 3: IPW and Naive ATE — Both Outcomes

In [5]:
def naive_ate(outcome, t, c):
    return t[outcome].mean() - c[outcome].mean()

def ipw_ate(outcome, t, c):
    return (np.average(t[outcome], weights=t['ipw']) -
            np.average(c[outcome], weights=c['ipw']))

for outcome, label, unit in [
    ('converted_to_paid',    'Conversion rate', 'pp'),
    ('first_month_revenue',  'Month 1 revenue', '$/user'),
]:
    n_ate = naive_ate(outcome, treated, control)
    i_ate = ipw_ate(outcome, treated, control)
    scale = 100 if unit == 'pp' else 1
    fmt   = '.2f' if unit == 'pp' else '.2f'
    print(f"{label}:")
    print(f"  Naive ATE: {n_ate*scale:{fmt}} {unit}")
    print(f"  IPW ATE:   {i_ate*scale:{fmt}} {unit}")
    print(f"  Difference (naive − IPW): {(n_ate - i_ate)*scale:{fmt}} → selection bias")
    print()

Conversion rate:
  Naive ATE: 12.08 pp
  IPW ATE:   8.06 pp
  Difference (naive − IPW): 4.02 → selection bias

Month 1 revenue:
  Naive ATE: 3.87 $/user
  IPW ATE:   2.56 $/user
  Difference (naive − IPW): 1.30 → selection bias



## Credibility Assessment

**Did SMD improve?**
Yes, substantially. The maximum |SMD| dropped from 0.606 (household_size) before
IPW to approximately 0.003 after IPW. All four numeric covariates are effectively
balanced post-weighting, well below the 0.1 threshold.

**Selection bias magnitude:**
The naive conversion ATE (~12pp) overstates the IPW-adjusted estimate (~8pp) by
approximately 4 percentage points. This gap is the estimated selection bias — users
who received the offer were more likely to convert anyway due to larger household
size and higher prior engagement. The naive estimate incorrectly attributes this
baseline advantage to the offer.

For revenue, naive ($3.87/user) overstates IPW ($2.56/user) by ~$1.31/user.

**Which estimate to present:** The IPW-adjusted ATE. The balance diagnostics confirm
the adjustment worked for observable covariates.

**Remaining assumption:** Unconfoundedness — all variables that influence both
offer assignment and conversion outcomes have been measured. Unobservable factors
(e.g., intent-to-subscribe at time of offer receipt) remain a limitation.
Treat IPW ATEs as directional estimates, not precise causal claims.
